# CE457-UTH

### Petros Grammatikou 3366
### Averkios Topkaroglou 3174
- Run all cels to see the outputs. Size too big to send with photos printed...

## Final Project

The goal of this project is to create a program that takes N images with partial overlap and returns the images stitched together into a panoramic image. You are given three example views of the same scene in the folder "images" to experiment with and verify your implementation. Follow these algorithmic steps in order to stitch N given images (repeat the process pairwise for all N images):

    - Load two consecutive images. 
	
    - Find common features between images using the SIFT algorithm. In particular, you should compute SIFT keypoints and corresponding descriptors (SIFT signatures) for each image. Match keypoints between overlapping images and display all the matching pairs for the given images. Use OpenCV’s built-in functions to assist in feature detection, descriptor extraction, and matching pair visualization.
	
    - After matching the keypoints, not all matches will be correct. Some will be outliers due to noise or mismatches. Use the RANSAC algorithm to filter out these outliers and retain only the inliers. Compute the homography matrix using the inliers. This matrix maps points from one image to their corresponding locations in the other image. Implement the RANSAC algorithm from scratch to estimate the homography matrix that maps one image onto the other. You are not allowed to use functions like cv2.findHomography or cv2.getPerspectiveTransform for RANSAC. However, you can use them for debugging or comparison. Report the number of inliers and display all inlier matches after removing outliers with RANSAC on the given pair.
	
	- Apply the homography transformation to align the images. Use the homography matrix computed in the previous step to warp one image (e.g., "img1.jpg") so that it aligns with the other image (e.g., "img2.jpg"). Use OpenCV's cv2.warpPerspective function to apply the transformation. Implementing your own warping function can earn bonus points in your final grade. Display the warped image.
	
	- Stitch transformed image "img1.jpg" and original image "img2.jpg" together. Merge images smoothly into a single panorama, using blending techniques. Experiment with at least three blending techniques (e.g., simple averaging, linear blending, or other OpenCV built-in methods, like Laplacian blending). Use the best-performing technique for the final panorama. Report which blending method works best for seamless image stitching. Display the stitched image after blending.


## Note

For estimating homography, ensure you write your own code for the constrained least squares minimization and RANSAC steps, as covered in Lecture 09 (Image Stitching). You may use built-in functions to compute the eigenvector corresponding to the smallest eigenvalue. 

## Deliverables
	
In addition to implementing the code, you should submit a report addressing the following points:

1. Provide a good description of your implementation.
2. Discuss your results for stitching the provided images ("img1.jpg" and "img2.jpg"). Explain any challenges faced when generalizing the solution to N images.	
3. What would happen if a random subset of pairs of matching keypoints was used for finding homography? Discuss the relationship between the number of pairs of matching keypoints used for finding homography and the accuracy of inferred homography.
4. Provide a comparison of blending techniques.


### Libraries

* [matplotlib](https://matplotlib.org/)
* [python-opencv](https://docs.opencv.org/4.x/d6/d00/tutorial_py_root.html)
* [numpy](https://numpy.org/)



In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import matplotlib as mpl
import imageio.v2 as imageio
import math

# SIFT
def sift_feature_matching(img1, img2):
    # grauscale works better
    gray1 = cv2.cvtColor(img1, cv2.COLOR_RGB2GRAY)
    gray2 = cv2.cvtColor(img2, cv2.COLOR_RGB2GRAY)

    sift = cv2.SIFT_create()

    # detect keypoints and compute ypografes
    kp1, des1 = sift.detectAndCompute(gray1, None)
    kp2, des2 = sift.detectAndCompute(gray2, None)

    # BFMatcher to match keypoints
    bf = cv2.BFMatcher(cv2.NORM_L2, crossCheck=True)
    matches = bf.match(des1, des2)

    # matches draw
    matched_img = cv2.drawMatches(img1, kp1, img2, kp2, matches[:], None, flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)

    # Extract point coordinates from the matches
    src_pts = np.float32([kp2[m.trainIdx].pt for m in matches])  # img2 pts
    dst_pts = np.float32([kp1[m.queryIdx].pt for m in matches])  # img1 pts

    return matched_img, src_pts, dst_pts, matches, kp1, kp2


# RANSAC implementation function
def compute_homography_ransac(src_pts, dst_pts, max_iters=1000, threshold=5.0):
    best_H = None
    best_inliers = []
    
    for _ in range(max_iters):
        indices = np.random.choice(len(src_pts), 4, replace=False)
        src_sample = src_pts[indices]
        dst_sample = dst_pts[indices]
        
        A = []
        b = []
        for i in range(4):
            x, y = src_sample[i]
            u, v = dst_sample[i]
            A.append([x, y, 1, 0, 0, 0, -u*x, -u*y])
            A.append([0, 0, 0, x, y, 1, -v*x, -v*y])
            b.append(u)
            b.append(v)
        
        A = np.array(A)
        b = np.array(b).reshape(-1, 1)
        
        if np.linalg.det(A.T @ A) == 0:
            continue  # Skip if matrix is singular
        
        H_vec = np.linalg.inv(A.T @ A) @ A.T @ b
        H = np.append(H_vec, [[1]], axis=0).reshape(3, 3)

        inliers = []
        for i, (x, y) in enumerate(src_pts):
            pred = H @ np.array([x, y, 1])
            pred /= pred[2]
            error = np.linalg.norm(pred[:2] - dst_pts[i])
            if error < threshold:
                inliers.append(i)

        if len(inliers) > len(best_inliers):
            best_inliers = inliers
            best_H = H
    
    if best_H is not None and len(best_inliers) >= 4:
        A = []
        b = []
        for i in best_inliers:
            x, y = src_pts[i]
            u, v = dst_pts[i]
            A.append([x, y, 1, 0, 0, 0, -u*x, -u*y])
            A.append([0, 0, 0, x, y, 1, -v*x, -v*y])
            b.append(u)
            b.append(v)
        
        A = np.array(A)
        b = np.array(b).reshape(-1, 1)
        
        if np.linalg.det(A.T @ A) != 0:
            H_vec = np.linalg.inv(A.T @ A) @ A.T @ b
            best_H = np.append(H_vec, [[1]], axis=0).reshape(3, 3)
    
    return best_H, best_inliers


# warp images on identical canvases to be ready for stitching
def warp_images(img1, img2, H):
    h1, w1 = img1.shape[:2]
    h2, w2 = img2.shape[:2]

    # Get corners of the two images
    corners1 = np.float32([[0, 0], 
                           [0, h1], 
                           [w1, h1], 
                           [w1, 0]]).reshape(-1, 1, 2)
    
    corners2 = np.float32([[0, 0], 
                           [0, h2], 
                           [w2, h2], 
                           [w2, 0]]).reshape(-1, 1, 2)

    # Find the corners first!
    warped_corners2 = cv2.perspectiveTransform(corners2, H)
    corners = np.concatenate((corners1, warped_corners2), axis=0)
    [xmin, ymin] = np.int32(corners.min(axis=0).ravel() - 0.5)
    [xmax, ymax] = np.int32(corners.max(axis=0).ravel() + 0.5)

    # Translation matrix
    t = [-xmin, -ymin]
    Ht = np.array([[1, 0, t[0]], 
                   [0, 1, t[1]], 
                   [0, 0, 1]])

    # Warp img2
    warped_img2 = cv2.warpPerspective(img2, Ht @ H, (xmax - xmin, ymax - ymin))

    # Create canvases
    canvas_size = (warped_img2.shape[0], warped_img2.shape[1], 3)
    canvas1 = np.zeros(canvas_size, dtype=np.uint8)
    canvas2 = np.zeros(canvas_size, dtype=np.uint8)

    # Place img1 on canvas1
    canvas1[t[1]:t[1] + h1, t[0]:t[0] + w1] = img1

    # Place warped_img2 canvas2
    canvas2 = warped_img2

    return canvas1, canvas2

# Performs Laplacian pyramid blending on two images using a mask
def laplacian_blending(img1, img2, mask, levels=6):
    
    # Convert input images and mask to float32
    img1, img2, mask = img1.astype(np.float32), img2.astype(np.float32), mask.astype(np.float32)

    # Generate Gaussian pyramids
    gp1, gp2, gpM = [img1], [img2], [mask]
    
    for _ in range(levels):
        img1 = cv2.pyrDown(gp1[-1])
        img2 = cv2.pyrDown(gp2[-1])
        mask = cv2.pyrDown(gpM[-1])

        gp1.append(img1)
        gp2.append(img2)
        gpM.append(mask)

    # Generate Laplacian pyramids
    lp1, lp2 = [gp1[-1]], [gp2[-1]]
    
    for i in range(levels, 0, -1):
        size = (gp1[i-1].shape[1], gp1[i-1].shape[0])  # Ensure correct target size
        
        L1 = cv2.subtract(gp1[i-1], cv2.resize(cv2.pyrUp(gp1[i]), size))
        L2 = cv2.subtract(gp2[i-1], cv2.resize(cv2.pyrUp(gp2[i]), size))
        
        lp1.append(L1)
        lp2.append(L2)

    # Ensure mask pyramid is properly resized
    for i in range(levels + 1):
        gpM[i] = cv2.resize(gpM[i], (lp1[i].shape[1], lp1[i].shape[0]))

    # Blend Laplacian pyramids using the mask
    blended_pyramid = []
    for i in range(levels + 1):
        blended = lp1[i] * gpM[i] + lp2[i] * (1 - gpM[i])
        blended_pyramid.append(blended)

    # Reconstruct the final blended image
    blended_image = blended_pyramid[0]
    for i in range(1, levels + 1):
        size = (blended_pyramid[i].shape[1], blended_pyramid[i].shape[0])
        blended_image = cv2.pyrUp(blended_image)
        blended_image = cv2.resize(blended_image, size)  
        blended_image = cv2.add(blended_image, blended_pyramid[i])
    
    blended = np.clip(blended_image, 0, 255).astype(np.uint8)
    return blended


# Blend the two images by averaging ONLY THE OVERLAPPING NON-ZERO REGIONS of the canvases
def stitch_images(canvas1, canvas2):
    stitched_image = np.zeros(canvas1.shape, dtype=np.uint8)

    for i in range(stitched_image.shape[0]):
        for j in range(stitched_image.shape[1]):
            if(canvas1[i, j].all() != 0 & canvas2[i, j].all() != 0):
                stitched_image[i, j] = 0.5*canvas1[i, j] + 0.5*canvas2[i, j]
            elif(canvas1[i, j].all() != 0):
                stitched_image[i, j] = canvas1[i, j]
            else:
                stitched_image[i, j] = canvas2[i, j]

    return stitched_image

# Blends using linear blending
def stitch_images_linear(canvas1, canvas2):
    
    # Create a mask for blending
    mask1 = (canvas1 > 0).astype(np.float32)
    mask2 = (canvas2 > 0).astype(np.float32)

    # Overlap region
    overlap = mask1 * mask2  

    # Generate weight map (smooth transition)
    weight1 = np.linspace(1, 0, canvas1.shape[1]).reshape(1, -1, 1)  # Left to right fade
    weight1 = np.tile(weight1, (canvas1.shape[0], 1, 3)) * overlap  # Apply only to overlap
    weight2 = 1 - weight1  # Opposite for img2

    # Linear blending
    stitched_image = (canvas1 * weight1 + canvas2 * weight2).astype(np.uint8)

    # Keep non-overlapping regions intact
    stitched_image[overlap == 0] = canvas1[overlap == 0] + canvas2[overlap == 0]

    return stitched_image

# Blends using laplacian fro above
def stitch_images_laplacian(canvas1, canvas2, levels=6):
    
    # Create blending mask (smooth transition in overlap)
    mask = np.zeros(canvas1.shape[:2], dtype=np.float32)
    mask[:, :canvas1.shape[1] // 2] = 1  # Left side fades in, right side fades out
    mask = cv2.merge([mask, mask, mask])  # Convert to 3 channels

    # Perform Laplacian blending
    stitched_image = laplacian_blending(canvas1, canvas2, mask, levels)

    return stitched_image

# Code to petsokops the image so it wont have black regions
# assume that in the middle of the image there is no black regions
# if there are 10 consecutive black pixels in the image the region is considered BLACK SO WE KILL IT
def petsokops(stitched_image):

    k = 0
    cut_above = 0
    for i in range(int(stitched_image.shape[0] / 2)):
        for j in range(stitched_image.shape[1]):
            if(stitched_image[i, j].all() == 0):
                k += 1
            else:
                k = 0
            if k == 10:
                k = 0
                cut_above = i


    k = 0
    l = 0
    cut_below = stitched_image.shape[1]
    for i in range(stitched_image.shape[0]-1, int(stitched_image.shape[0] / 2), -1):
        l+=1
        for j in range(stitched_image.shape[1]):
            if(stitched_image[i, j].all() == 0):
                k += 1
            else:
                k = 0
            if k == 10:
                k = 0
                cut_below = l
                

    # cut out to the dimensions limits 
    panorama_mine = stitched_image[cut_above:(stitched_image.shape[0] - cut_below), 0:stitched_image.shape[1]-3, :]

    return panorama_mine

In [ ]:
# Step 1: Load Images
img1 = imageio.imread('images/img1.jpg')
img2 = imageio.imread('images/img2.jpg')
img3 = imageio.imread('images/img3.jpg')

#alternate read with cv2
#img1 = cv2.imread('images/img1.jpg')

plt.figure(figsize=(13,13))
plt.subplot(131); plt.imshow(img1)  
plt.subplot(132); plt.imshow(img2)
plt.subplot(133) ;plt.imshow(img3)


In [ ]:
# Step 2: Apply SIFT and find maches
matched_img, src_pts, dst_pts, matches, kp1,kp2 = sift_feature_matching(img1, img2)

# Display the matching result
plt.figure(figsize=(15, 10))
plt.imshow(matched_img)
plt.title("Feature Matching with SIFT: img1-img2")
plt.show()

In [ ]:
# Step 3: Compute homography using RANSAC
H, inliers = compute_homography_ransac(src_pts, dst_pts)
print(f"Number of inliers: {len(inliers)}")

# Draw inlier matches
inlier_matches = [matches[i] for i in inliers]
inlier_img = cv2.drawMatches(img1, kp1, img2, kp2, inlier_matches, None,
                            flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
plt.figure(figsize=(15, 5))
plt.imshow(inlier_img)
plt.title(f'Step 3: Inlier Matches ({len(inliers)} found)')
plt.show()

In [ ]:
# Step 4: Apply homography and Warp the images
canvas1, canvas2 = warp_images(img1, img2, H)

# Step 5: Stich and blend images
# Stitch the images
stitched_image = stitch_images(canvas1, canvas2)

# Stitch the images using linear blending
stitched_image_linear = stitch_images_linear(canvas1, canvas2)

# Stitch the images using Laplacian blending
stitched_image_laplacian = stitch_images_laplacian(canvas1, canvas2, levels=6)


# Display the final stitched image
plt.figure(figsize=(15, 5))
plt.imshow(stitched_image)
plt.title("Stitched Image")
plt.show()

# Display the final stitched image
plt.figure(figsize=(10, 5))
plt.imshow(stitched_image_linear)
plt.title("Stitched Image with Linear Blending")
plt.show()

# Display the final stitched image
plt.figure(figsize=(10, 5))
plt.imshow(stitched_image_laplacian)
plt.title("Stitched Image with Laplacian Blending")
plt.show()


In [ ]:
# Starting from the top to stich the already stiched image 1-2 with image 3
# This part is realy SLOWWWW. Give it some time!
# Apply sift
mtc_img_avg, src_pts_avg, dst_pts_avg, matches_avg, kp1_avg, kp2_avg = sift_feature_matching(stitched_image, img3)
mtc_img_lin, src_pts_lin, dst_pts_lin, matches_lin, kp1_lin, kp2_lin = sift_feature_matching(stitched_image_linear, img3)
mtc_img_lap, src_pts_lap, dst_pts_lap, matches_lap, kp1_lap, kp2_lap = sift_feature_matching(stitched_image_laplacian, img3)

# Compute homography using RANSAC
H_avg, inliers_avg = compute_homography_ransac(src_pts_avg, dst_pts_avg, max_iters=1000, threshold=3.0)
H_lin, inliers_lin = compute_homography_ransac(src_pts_lin, dst_pts_lin, max_iters=1000, threshold=3.0)
H_lap, inliers_lap = compute_homography_ransac(src_pts_lap, dst_pts_lap, max_iters=1000, threshold=3.0)


In [ ]:
# Warp the images
canvas1_avg, canvas2_avg = warp_images(stitched_image, img3, H_avg)
canvas1_lin, canvas2_lin = warp_images(stitched_image_linear, img3, H_lin)
canvas1_lap, canvas2_lap = warp_images(stitched_image_laplacian, img3, H_lap)


# Stitch the images
stitched_image = stitch_images(canvas1_avg, canvas2_avg)

# Stitch the images using linear blending
stitched_image_linear = stitch_images_linear(canvas1_lin, canvas2_lin)

# Stitch the images using laplacian blending
stitched_image_laplacian = stitch_images_laplacian(canvas1_lap, canvas2_lap)

# Display the final stitched image
plt.figure(figsize=(10, 5))
plt.imshow(stitched_image)
plt.title("Stitched Image")
plt.show()

# Display the final stitched image
plt.figure(figsize=(10, 5))
plt.imshow(stitched_image_linear)
plt.title("Stitched Image with Linear Blending")
plt.show()

# Display the final stitched image
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 5))
plt.imshow(stitched_image_laplacian)
plt.title("Stitched Image with Laplacian Blending")
plt.show()

In [ ]:
panorama_mine = petsokops(stitched_image) 
panorama_mine_linear = petsokops(stitched_image_linear) 
panorama_mine_laplacian = petsokops(stitched_image_laplacian) 

# Display the final stitched image
plt.figure(figsize=(15, 10))
plt.imshow(panorama_mine)
plt.title("Final Stitched Panorama (avg blending)")
plt.show()
plt.figure(figsize=(15, 10))
plt.imshow(panorama_mine_linear)
plt.title("Final Stitched Panorama (linear blending)")
plt.show()
plt.figure(figsize=(15, 10))
plt.imshow(panorama_mine_laplacian)
plt.title("Final Stitched Panorama (laplacian blending)")
plt.show()

In [ ]:
# FOR VALIDATION WE COMPARE WITH THE OUTCOME OF AN EXISTING LIBRARY
# NEED YOU TO RUN:
# !pip install stitching
# IF NOT INSTALLED
import stitching.stitcher as st
    
img1 = imageio.imread('images/img1.jpg')
img2 = imageio.imread('images/img2.jpg')
img3 = imageio.imread('images/img3.jpg')

stitcher = st.Stitcher()
panorama = stitcher.stitch([img1, img2, img3])

plt.figure(figsize=(15, 10))
plt.imshow(panorama)
plt.title("PANORAMA BUILT IN")
plt.show()

# plot the best of ours to compare 
plt.figure(figsize=(15, 10))
plt.imshow(panorama_mine_laplacian)
plt.title("PANORAMA IMPLEMENTED")
plt.show()

# Conclusion
Σε αυτό το project εξετάσαμε την διαδικασία πίσω από το image stiching. Συγκεκριμένα είδαμε το πως 3 (ή και παραπάνω) εικόνες μπορούμε να τις συρράψουμε σε ένα συνολικό πανόραμα. Για να υλοποιηθεί λοιπόν αυτό γράψαμε τις συναρτήσεις που φαίνονται στο πρώτο cerll αυτού του αρχείο ακολουθώντας τα παρακάτω βήματα.
## Step 1: Load images
Φόρτωση των εικόνων μου θέλουμε να συρράψουμε.
## Step 2: Compute Matches
Ξεκινόντας πάντα από την αριστερή εικόνα με την συνάρτηση sift_feature_matching εντωπίζουμε τα σημεία ενδιαφέροντος μέσω του SIFT detector και χρησιμοποιώντας τον αλγόριθμο BFMatcher αντιστοιχίζουμε τα keypoints με βάση τις περιγραφές τους. Τέλος με την συνάρτηση drawMatches σχεδιάζουμε αυτά τα matches πάνω στις 2 εικόνες.
## Step 3: Compute homography using RANSAC
Τα matches αυτά δεν είναι όλα αποδεκτά (inliers) και πρέπει να φιλτραριστούν. Αυτό το πετυχαίνουμε υπολογίζοντας τον πίνακα homography μέσα από την συνάρτηση compute_homography_ransac. Συγκεκριμένα, ακολουθώντας τον αλγόριθμο RANSAC, επιλέγουμε 4 τυχαία ζευγάρια από τα matches και υπολογίζουμε το homography τους με την μέθοδο των ελάχιστων τετραγώνων. Έπειτα εξετάζουμε πόσα από τα αυτά είναι inliers μέσω του συγκεκριμένου homopgraphy και επαναλαμβάνουμε την διαδικασία συνολικά 1000 φορές. Στο τέλος κρατάμε τον homography που μας έδωσε τον μεγαλύτερο αριθμό από inliers.
## Step 4: Apply homography and Warp the images
Σε αυτό το βήμα χρησιμοποιούμε τον πίνακα homography που βρήκαμε προηγουμένος για να μεταφέρουμε τις εικόνες σε κοινό coordinate frame. Ξεκινώντας, όπως και στην θεωρία, από τις γωνίες της δεξιάς εικόνας την μεταβιβάζουμε στο coordinate frame της αριστερά εικόνας μέσω του translation matrix και τις τοποθετούμε σε καμβάδες ίδιων διαστάσεων για να γίνει η συρραφή τους ευκολότερη στο επόμενο βήμα.
## Step 5: Stich and blend images
Τελευταίο μας βήμα είναι η συρραφή των εικόνων μεταξύ τους. Το να τοποθετήσουμε όμως απλά την μία εικόνα πάνω στην άλλη δεν μας δίνει αποδεκτό αποτέλεσμα καθώς στα σημεία της συρραφής η μετάβαση είναι πολύ απότομη. Για το λόγω αυτό εξετάσαμε 3 μεθόδους blending εκείνης της περιοχής ώστε η μετάβαση να γίνει πιο ομαλή.
### Average Blending:
Για την υλοποίηση του average blending προτιμήσαμε να κάνουμε μία δικιά μας υλοποίηση, stich_image, η οποία συγχωνεύει με βάρη 0.5 τις εικόνες μόνο στα σημεία όπου έχουν επικάλυψη μεταξύ τους ενώ τις αφήνει ίδιες αλλού. Η μέθοδος αυτή είναι πολύ απλή, υπολογιστικά αποδοτική και παράγει πολύ ικανοποιητικά αποτελέσματα.
### Linear Blending:
Η συνάρτηση stitch_images_linear συγχωνεύει και πάλι τις εικόνες στην περιοχή που υπάρχει overlap με την διαφορά ότι τα βάρη αυτή την φορά μεταβάλονται γραμμικά σε αντίθεση με το average blending οπου τα βάρη είναι σταθερά. Το αποτέλεσμα είναι μια ποιο  smooth συρραφή στις εικόνες συγκριτικά με την πρώτη μέθοσο ωστόσο για κάποιο λόγο φαίνεται έντωνα στο πανόραμα η αριστερή πλευρά της δεξιάς εικόνας. Στα θετικά είναι πως το linear blending είναι αρκετά αποδοτικό υπολογιστικά και πολύ πιο απλό συγκριτικά με το Laplacian Blending που θα δούμε στην συνέχεια.
### Laplacian Blending:
Τελευταία και πιο περίπλοκη μέθοδος είναι το laplacian blending. Εδώ μέσα από την συνάρτηση laplacian_blending δημιουργούμε τις Laplacian πυραμίδες για τις 2 εικόνες υπολογίζοντας πρώτα τις Gaussian ππυραμίδες και τις χρησιμοποιούμε στην συνάρτηση stitch_images_laplacian για την σταδιακή συγχώνευση των εικόνων. Η μέθοδος αυτή μας δίνει το καλύτερο τελικό αποτέλεσμα καθώς η μετάβαση μεταξύ των 2 εικόνων είναι πολύ ομαλή διατηρώντας τις περισσότερες λεπτομέρειες στην overlaping περιοχή. Βέβαια ο υπολογιστικός φόρτος είναι αυξημένος σε αυτή την περίπτωση και στις πρώτες μας προσπάθειες παρατηρήσαμε προβληματα με την δημιουργία artifacts (κόκκινα, κίτρινα και κενά pixels) στην περιοχή της μάσκας καθώς δεν ήταν σωστά διαμορφωμένη.
## Περικοπή και Σύγκριση αποτελεσμάτων
Προαιρετικά φτιάξαμε την συνάρτηση petsokops η οποία ανιχνεύει το "ανωτερο" και το "κατώτερο" σημείο του πανοράματος και κόβει τις μαύρες και ακατέργαστες  περιοχές του καμβά στον οποίο βάλαμε πάνω τις εικόνες μετά το warping.  Υποθέτουμε οτι η περιοχή θεωρείτε μαύρη εφόσον είχε 10 συνεχόμενα μαύρα pixel και μπορεί να περικοπεί. 

Τέλος, για δική μας επιβεβαίωση, θελήσαμε να συγκρίνουμε το τελικό μας πανόραμα με αυτό που μας δίνει η built in βιβλιοθήκη stitching της python. Βλέπουμε πως το δικό μας implemetation μας δίνει ένα πιο λεπτομερές πανόραμα με μεγαλύτερη ευκρίνεια και διαστάσεις ενώ το Laplacian Blending που εκτελέσαμε συρράφει σχεδόν τέλεια τις εικόνες μας. Σίγουρα όμως η built in μέθοδος είναι πιο εύχριστη από το δικό μας implemetation καθώς με 2 σειρές κώδικα δημιουργείτε το πανόραμα όσων εικόνων δώσουμε σαν ορίσματα ενώ στην δική μας περίπτωση περιοριζόμαστε στο να συρράφουμε τις εικόνες μας πάντα από τα αριστερά προς τα δεξιά και πάντα την 1η με την 2η και μετά το αποτέλεσμά τους με την 3η.